# Interactive choropleth — ICSA / WICSA / ECSA affiliations by year

Reads `data/raw/icsa_affiliations.json`, counts author-institution rows by
country for each year, and shows a world choropleth with a **year slider**.
The colour scale is locked across years so maps are comparable.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

## 1. Config

In [2]:
INPUT_FILE = Path("../../data/raw/icsa_affiliations.json")
CMAP       = "YlOrRd"
TITLE      = "ICSA / WICSA / ECSA — author-institution rows by country"

## 2. Load records & build `{year: {iso_a3: count}}`

In [3]:
with open(INPUT_FILE) as f:
    records = json.load(f)

df = pd.DataFrame(records)
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df["country_code"] = df["country_code"].str.upper().str.strip()
df = df.dropna(subset=["year", "country_code"])

# world geometry uses ISO alpha-3; the data has alpha-2 -> bridge them
try:
    import pycountry
    def a2_to_a3(a2):
        try:
            return pycountry.countries.get(alpha_2=a2).alpha_3
        except Exception:
            return None
except ImportError:
    A2_TO_A3 = {
        "AR":"ARG","AU":"AUS","AT":"AUT","BE":"BEL","BR":"BRA","CA":"CAN",
        "CH":"CHE","CL":"CHL","CN":"CHN","CO":"COL","CZ":"CZE","DE":"DEU",
        "DK":"DNK","EG":"EGY","ES":"ESP","FI":"FIN","FR":"FRA","GB":"GBR",
        "GR":"GRC","HK":"HKG","HU":"HUN","IE":"IRL","IL":"ISR","IN":"IND",
        "IR":"IRN","IT":"ITA","JP":"JPN","KR":"KOR","LU":"LUX","MX":"MEX",
        "MY":"MYS","NL":"NLD","NO":"NOR","NZ":"NZL","PL":"POL","PT":"PRT",
        "RO":"ROU","RU":"RUS","SE":"SWE","SG":"SGP","TR":"TUR","TW":"TWN",
        "UA":"UKR","US":"USA","ZA":"ZAF",
    }
    a2_to_a3 = lambda a2: A2_TO_A3.get(a2)
    print("pycountry not found - using built-in alpha-2->3 lookup (pip install pycountry for full coverage).")

df["iso_a3"] = df["country_code"].apply(a2_to_a3)
unmatched = sorted(df.loc[df["iso_a3"].isna(), "country_code"].unique())
if unmatched:
    print("Could not map to alpha-3:", unmatched)
df = df.dropna(subset=["iso_a3"])

counts_by_year = df.groupby(["year", "iso_a3"]).size().reset_index(name="count")
data = {
    int(y): dict(zip(g["iso_a3"], g["count"]))
    for y, g in counts_by_year.groupby("year")
}

YEARS = sorted(data)
VMAX  = int(counts_by_year["count"].max())   # lock colour scale across years
print(f"{len(df):,} rows | years {YEARS[0]}-{YEARS[-1]} | max country-year count = {VMAX}")

1,828 rows | years 2010-2025 | max country-year count = 63


## 3. Load world geometry

`gpd.datasets` was removed in geopandas 1.x, so load Natural Earth from its source.

In [ ]:
# Natural Earth, cached locally so it downloads only once (no slow re-fetch).
import urllib.request

url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
CACHE  = Path("../../data/raw/ne_110m_admin_0_countries.zip")

if not CACHE.exists():
    print("downloading Natural Earth (one time)...")
    urllib.request.urlretrieve(url, CACHE)

world = gpd.read_file(CACHE)
world = world.rename(columns={"ADM0_A3": "iso_a3"})[["iso_a3", "geometry"]]
print(f"world geometry: {len(world)} countries (cached at {CACHE})")

world geometry: 177 countries (cached at ../../data/raw/ne_110m_admin_0_countries.zip)


## 4. Interactive map — drag the year slider

In [ ]:
def plot_world(year):
    year_counts = data.get(year, {})
    w = world.copy()
    w["value"] = w["iso_a3"].map(year_counts)

    fig, ax = plt.subplots(figsize=(14, 7))
    fig.patch.set_facecolor("white")

    # countries with no data for this year
    w[w["value"].isna()].plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.4)

    # choropleth for countries that have data
    layer = w[w["value"].notna()]
    if not layer.empty:
        layer.plot(
            column="value", ax=ax, cmap=CMAP,
            edgecolor="white", linewidth=0.4,
            vmin=0, vmax=VMAX, legend=True,
            legend_kwds={"label": "Author-institution rows", "shrink": 0.6},
        )

    total = int(sum(year_counts.values()))
    ax.set_title(f"{TITLE}\n{year}  -  {total} rows", fontsize=13, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

interact(
    plot_world,
    year=widgets.IntSlider(value=YEARS[0], min=YEARS[0], max=YEARS[-1],
                           step=1, description="Year:", continuous_update=False),
);

interactive(children=(IntSlider(value=2010, continuous_update=False, description='Year:', max=2025, min=2010),…